## Population assignment

Attach demographic load to network.

In [ ]:
# Set parent as root and import config
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
import configs.simulation_config as cfg

import osmnx as ox

In [ ]:
def detect_exit_nodes(nodes_df):
    """
    Identify two exit nodes for out-of-city evacuation.
    Currently uses geographic heuristic.
    """

    ne_id = (nodes_df["x"] + nodes_df["y"]).idxmax()
    sw_id = (nodes_df["x"] + nodes_df["y"]).idxmin()

    return [ne_id, sw_id]

In [ ]:
graph = ox.load_graphml("../data/processed/hatyai_graph_with_pop.graphml")
nodes, edges = ox.graph_to_gdfs(graph)

exit_nodes = detect_exit_nodes(nodes)

for node in graph.nodes:
    graph.nodes[node]["evac_dest"] = "shelter"
    graph.nodes[node]["is_out_of_city"] = False

if cfg.OUT_CITY_FRAC > 0:
    for node in exit_nodes:
        graph.nodes[node]["evac_dest"] = "out_city"
        graph.nodes[node]["is_out_of_city"] = True
    
    print("Exit nodes:", exit_nodes)

ox.save_graphml(graph, "../data/processed/hatyai_graph_with_dest.graphml")